# Libraries

In [45]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from scipy.stats import spearmanr


# Contexte

Les prix de marchés de l'électricité dépendent de facteurs multiples. La première chose à avoir en tête c'est qu'il existe des multitudes de marchés (à terme, day-ahead, intraday, balancing, ancillaires, etc). Dans ce proje nous nous contenterons des prix à terme de maturité 24h. Ces produits sont donc la livraison d'1MW d'électricité pour une durée de 24h. 

Le but ici n'est pas de prédire le prix du jour mais la variation journalière (prix observé pour le produit en J-1 vs J).

À présent nous allons parler de la formation de ces prix. Il est important de comprendre que les prix à terme ne sont que le reflet de la moyenne pondérée de l'espérance des prix spots. Or ces derniers changent toutes les heures en fonction de l'équilibre offre/demande (il est à noté que depuis le 1er octobre 2025 on est passé à une granularité de 15 min). Cet équilibre offre/demande permet la constitution d'un prix, d'un côté les consommateurs et d'un autres les producteurs. Les consommateurs étant très inélastiques aux prix, ils sont très souvent "price taker", c'est-à-dire que le prix sur une heure donner est le coût marginal subit par la dernière centrale électrique appelé pour équilibrer le réseau (principe de "merit order"). 

Or depuis quelques années, les marchés de l'électricité ont été impactés par de nombreux événements. Premièrement en 2022, après l'invasion de l'Ukraine par la Russie, on a observé une augmentation énorme sur les prix du gaz, passant de prix ossilant autour de 20€/MWh (pour le TTF, indice de prix de référence en EUROPE) à plus de 350€/MWh au pic de la crise. Comme expliqué précédemment, le prix de l'électricité est fixé par la dernière unité de production appelé, malheuresement quand la dernière unité appelé est une centrale à gaz, les prix spots s'envolent. Sachant qu'en général une centrale à gaz à un rendement entre 35% (cycle simple) et 60% (cycle combiné), il faut donc 2 MWh (en moyenne) de gaz pour produire 1MWh d'électricité. Ajouté à cela les crédits carbone à sourcer pour couvrir les émissions qu'engendre la centrale à gaz les prix se sont envoler à plus de 1000€/MWh sur certaines heures. 

On comprend donc un premier problème. Les prix de l'électricité dépendent de la dernière unité appelé pour équilibrer le réseau. 

Depuis 5 ans, on a aussi assisté à un développement exponentiel des capacités de production renouvelables (nous parlons ici du solaire et de l'éolien, l'hydraulique étant, surtout en France, déjà a son plein potentiel d'exploitation). Ce développement massif d'ENR a un impact sur les prix surtout en milieu de journée (moment où le soleil est à son zénith et donc que le solaire produit le plus). Cette production massive d'électricité solaire augmente drastiquement l'offre électrique à un coût marginal quasiment nul. De plus, sur ces heures de milieu de journée la demande électrique est très souvent la plus basse de la journée, les gens étant au travail ne consomme pas chez eux, il fait souvent plus chaud et donc les besoins en chauffage soint moindres, etc. Ces deux phénomènes combinés, entraîne des prix largement négatifs surtout en période de "non stress" électrique, c'est-à-dire au printemps et en été. 

On voit l'émergence d'une seconde problématique qui est le non alignement entre production abondante d'ENR et demande électrique. 

Si on se replace dans le contexte de la crise de 2022, un autre problème a été mis en exergue, celui de la disponibilité du parc nucléaire français. Dans les années 1970, un grand plan de construction de centrales nucléaires a été développé afin de contrecarrer les impacts liés aux variations de prix du pétrole (le fameux, "En France, on a pas de pétrole mais on a des idées"). Ce qui faisait l'une des fierté de la France a été l'un des facteurs aggravant de la crise de 2022. En effet, durant cette crise la disponibilité du parc nucléaire français a été à son plus bas historique (les opérations de maintenance ainsi que les alertes sur certaines centrale se multipliant), de ce fait on a du importer plus d'électricité depuis nos voisins et faire tourner des centrales à gaz qui ne tournaient pas avant. 

On aperçoit donc ici un troisième problème (qui va de pair avec le deuxième problème) le mix électrique du pays, mais aussi sa capacité à équilibrer son réseau par les échanges transfontaliers. Ici nous ne nous concentrerons que sur l'Allemagne et la France qui sont des pays assez largement connecté à leurs voisins au niveau électrique. Ce problème est donc moins importants que pour des pays comme l'Espagne (cf le blackout d'avril 2025). 

Un dernier point à aborder pour finir cette mise en contexte est la différence qui existe entre le mix électrique français et le mix électrique allemand. En France, comme expliqué un peu plus haut, on a un mix électrique qui repose principalement sur du nucléaire (entre 65% et 70%, ENR entre 25 et 30%, le reste étant fossiles à des fins d'équilibrage de réseau en période de pointe), cela nous permet d'avoir une production dite de "base" prévisible, résiliente mais aussi (petite fierté nationale) assez pilotable afin de diminuer la production nucléaire sur les heures de production d'ENR afin de ne pas surcharger le réseau. En Allemagne, la politique énergétique menée depuis 10 ans a été basé sur une suppression des capacités nucléaires (0 centrale en 2023), ainsi qu'un développement massif d'ENR (mix composé en 2025 d'environ 50% d'ENR, 35% charbon et 15% de gaz). Ce plan, qui paraît plus respectueux de l'environnement cache en fait un gros problème, l'intermittence. En période de faible ensoleillement ainsi que de manque de vent, le réseau est très vite saturé, et les besoins électriques sont colossaux. C'est à ce moment là que les prix grimpent avec des importations qui explosent. 

Après ce bref tour d'horizon, on comprend bien que les marchés de l'électricité sont complexes et que ce qui forme le prix dépend d'énormément de facteurs spécifiques à chacun des pays. De ce fait, utilisé un seul modèle pour chaque pays serait omettre les différences intrinsèque qui existent entre chacun d'eux. 

# Data Management

## Data Import

In [90]:
# Importation des données
X_train = pd.read_csv("X_train.csv")
y_train = pd.read_csv("y_train.csv")
X_test = pd.read_csv("X_test.csv")
y_test = pd.read_csv("y_test_example.csv")

In [71]:
# On classe les colonnes de X_train par thème
columns_dict = {
    "to_keep" : [
        'ID', 'DAY_ID', 'COUNTRY'
        ],
        
    "production" : [
        'DE_GAS', 'FR_GAS', 'DE_COAL', 'FR_COAL', 'DE_HYDRO', 'FR_HYDRO', 'DE_NUCLEAR', 'FR_NUCLEAR', 'DE_SOLAR', 'FR_SOLAR', 'DE_WINDPOW', 'FR_WINDPOW', 'DE_LIGNITE'
       ], 
       
    "consumption" : [
        'DE_CONSUMPTION', 'FR_CONSUMPTION', 'DE_RESIDUAL_LOAD', 'FR_RESIDUAL_LOAD'
        ], 

    "exchange" : [
        'DE_FR_EXCHANGE', 'FR_DE_EXCHANGE', 'DE_NET_EXPORT', 'FR_NET_EXPORT','DE_NET_IMPORT', 'FR_NET_IMPORT'
       ],

    "weather" : [
        'DE_RAIN', 'FR_RAIN', 'DE_WIND','FR_WIND', 'DE_TEMP', 'FR_TEMP'
        ], 

    "commodity_price" : [
        'GAS_RET', 'COAL_RET', 'CARBON_RET'
        ]
}


## QRT Benchmark check

In [91]:
from sklearn.linear_model import LinearRegression
from scipy.stats import spearmanr

lr = LinearRegression()

X_train_clean = X_train.drop(['COUNTRY'], axis=1).fillna(0)
Y_train_clean = y_train['TARGET']

lr.fit(X_train_clean, Y_train_clean)

output_train = lr.predict(X_train_clean)

print(
    "Corrélation (Spearman) pour les données d'entraînement : {:.1f}%".format(
        100 * spearmanr(Y_train_clean, output_train).correlation
    )
)

X_test_clean = X_test.drop(['COUNTRY'], axis=1).fillna(0)

Y_test_submission = X_test[['ID']].copy()
Y_test_submission['TARGET'] = lr.predict(X_test_clean)

eval_df = pd.merge(
    y_test[['ID', 'TARGET']],
    Y_test_submission,
    on='ID',
    suffixes=('_true', '_pred')
)

print(
    "Corrélation (Spearman) pour les données de test : {:.1f}%".format(
        100 * spearmanr(eval_df['TARGET_true'], eval_df['TARGET_pred']).correlation
    )
)

Corrélation (Spearman) pour les données d'entraînement : 27.9%
Corrélation (Spearman) pour les données de test : -0.6%


## Data cleaning

In [37]:
# Affichage du nombre de valeurs manquantes par colonne
print(len(X_train))
for key in columns_dict.keys():
    print(f"{key}: {X_train[columns_dict[key]].isna().sum()}")

print(len(X_train.dropna(how="any")))

1494
to_keep: ID         0
DAY_ID     0
COUNTRY    0
dtype: int64
production: DE_GAS        0
FR_GAS        0
DE_COAL       0
FR_COAL       0
DE_HYDRO      0
FR_HYDRO      0
DE_NUCLEAR    0
FR_NUCLEAR    0
DE_SOLAR      0
FR_SOLAR      0
DE_WINDPOW    0
FR_WINDPOW    0
DE_LIGNITE    0
dtype: int64
consumption: DE_CONSUMPTION      0
FR_CONSUMPTION      0
DE_RESIDUAL_LOAD    0
FR_RESIDUAL_LOAD    0
dtype: int64
exchange: DE_FR_EXCHANGE     25
FR_DE_EXCHANGE     25
DE_NET_EXPORT     124
FR_NET_EXPORT      70
DE_NET_IMPORT     124
FR_NET_IMPORT      70
dtype: int64
weather: DE_RAIN    94
FR_RAIN    94
DE_WIND    94
FR_WIND    94
DE_TEMP    94
FR_TEMP    94
dtype: int64
commodity_price: GAS_RET       0
COAL_RET      0
CARBON_RET    0
dtype: int64
1276


In [38]:
# On ne garde que les lignes sans valeurs manquantes, triées par ordre chronologique
X_train_clean = (
    X_train
    .dropna(how="any")
    .sort_values(by="DAY_ID")
    .reset_index(drop=True)
)

y_train_clean = y_train[y_train["ID"].isin(X_train_clean["ID"])].reset_index(drop=True)

X_test_clean = (
    X_test
    .dropna(how="any")
    .sort_values(by="DAY_ID")
    .reset_index(drop=True)
)

y_test_clean = y_test[y_test["ID"].isin(X_test_clean["ID"])].reset_index(drop=True)


In [61]:
cols_to_scale = [
    col for col in X_train_clean.columns 
    if col not in columns_dict["to_keep"]
]

scaler = StandardScaler()

X_train_scaled = X_train_clean.copy()
X_train_scaled[cols_to_scale] = scaler.fit_transform(X_train_clean[cols_to_scale])

X_test_scaled = X_test_clean.copy()
X_test_scaled[cols_to_scale] = scaler.transform(X_test_clean[cols_to_scale])

# Colonnes FR/DE
fr_cols = list(dict.fromkeys(columns_dict["to_keep"] + [col for col in X_train_scaled.columns if "FR" in col]))
de_cols = list(dict.fromkeys(columns_dict["to_keep"] + [col for col in X_train_scaled.columns if "DE" in col]))

# Séparation train
X_train_FR = (
    X_train_scaled[fr_cols]
    .query("COUNTRY == 'FR'")
    .reset_index(drop=True)
)
y_train_FR = (
    y_train_clean[y_train_clean["ID"].isin(X_train_FR["ID"])]
    .reset_index(drop=True)
)

X_train_DE = (
    X_train_scaled[de_cols]
    .query("COUNTRY == 'DE'")
    .reset_index(drop=True)
)
y_train_DE = (
    y_train_clean[y_train_clean["ID"].isin(X_train_DE["ID"])]
    .reset_index(drop=True)
)

# Séparation test
X_test_FR = (
    X_test_scaled[fr_cols]
    .query("COUNTRY == 'FR'")
    .reset_index(drop=True)
)
y_test_FR = (
    y_test_clean[y_test_clean["ID"].isin(X_test_FR["ID"])]
    .reset_index(drop=True)
)

X_test_DE = (
    X_test_scaled[de_cols]
    .query("COUNTRY == 'DE'")
    .reset_index(drop=True)
)
y_test_DE = (
    y_test_clean[y_test_clean["ID"].isin(X_test_DE["ID"])]
    .reset_index(drop=True)
)

# Jeux finaux
train_dict = {
    "ALL": pd.merge(X_train_scaled, y_train_clean, on="ID"),
    "FR": pd.merge(X_train_FR, y_train_FR, on="ID"),
    "DE": pd.merge(X_train_DE, y_train_DE, on="ID")
}

test_dict = {
    "ALL": pd.merge(X_test_scaled, y_test_clean, on="ID"),
    "FR": pd.merge(X_test_FR, y_test_FR, on="ID"),
    "DE": pd.merge(X_test_DE, y_test_DE, on="ID")
}


# First Models

In [88]:
models = {
    "Linear": LinearRegression(),
    "Ridge (alpha = 1)": Ridge(alpha=1.0),
    "Lasso": Lasso(alpha=0.1, max_iter=10000)
}

cols_to_drop = columns_dict["to_keep"]  # Colonnes à ne pas utiliser pour l'entraînement

results = []

for dataset_name in train_dict:
    data_train = train_dict[dataset_name]
    data_test = test_dict[dataset_name]

    y_train = data_train["TARGET"]
    X_train = data_train.drop(columns=["TARGET"] + cols_to_drop, errors="ignore")

    y_test = data_test["TARGET"]
    X_test = data_test.drop(columns=["TARGET"] + cols_to_drop, errors="ignore")

    for model_name, model in models.items():
        model.fit(X_train, y_train)

        y_pred_train = model.predict(X_train)
        y_pred_test = model.predict(X_test)

        results.append({
            "Dataset": dataset_name,
            "Model": model_name,
            "Spearman Train": spearmanr(y_train, y_pred_train).correlation,
            "Spearman Test": spearmanr(y_test, y_pred_test).correlation
        })

results_df = (
    pd.DataFrame(results)
    .sort_values(by="Spearman Test", ascending=False)
    .reset_index(drop=True)
)

results_df

,Dataset,Model,Spearman Train,Spearman Test
0,DE,Lasso,0.361798,0.024263
1,DE,Ridge (alpha = 1),0.414503,0.003093
2,DE,Linear,0.413226,0.001465
3,ALL,Lasso,0.227023,-0.018582
4,ALL,Ridge (alpha = 1),0.295076,-0.019175
5,ALL,Linear,0.294609,-0.021457
6,FR,Ridge (alpha = 1),0.174670,-0.029949
7,FR,Linear,0.170818,-0.033985
8,FR,Lasso,0.132695,-0.045495
